# Task 3.1 — Two-Component Ablation Study
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification (Tu & Lin, ICML 2010)  
**Student:** Vaishnav Verma (230160)

This notebook ablates two distinct components of the OSSVR method, one at a time, and measures the impact on average misclassification cost.

In [1]:
# === Configuration & Seeds ===
import numpy as np
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

TEST_SIZE = 0.3
C_REG = 100.0
GAMMA = 0.5

COST_MATRIX = np.array([
    [0, 1, 5],
    [2, 0, 3],
    [10, 1, 0],
])

/Users/vaishnavverma/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/matplotlib-mchfk1ys because there was an issue with the default path (/Users/vaishnavverma/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [2]:
# === Data Setup (same as Task 2) ===
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from scipy.optimize import minimize

wine = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)
y = wine.target
K = len(np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

regret_train = COST_MATRIX[y_train]
regret_test = COST_MATRIX[y_test]

def avg_cost(y_true, y_pred, C):
    return np.mean(C[y_true, y_pred])

print(f"Data: {X_train.shape[0]} train, {X_test.shape[0]} test, {K} classes")

Data: 124 train, 54 test, 3 classes


In [3]:
# === Full OSSVR (baseline for ablation — dual QP, no bias) ===
import cvxpy as cp

class OneSidedSVR:
    def __init__(self, C=1.0, gamma='scale'):
        self.C = C
        self.gamma = gamma
        self.alpha_ = None
        self.X_train_ = None
        self.gamma_value_ = None
    
    def _compute_gamma(self, X):
        if self.gamma == 'scale':
            return 1.0 / (X.shape[1] * X.var())
        return self.gamma
    
    def fit(self, X, r):
        N = X.shape[0]
        self.X_train_ = X.copy()
        self.gamma_value_ = self._compute_gamma(X)
        K_mat = rbf_kernel(X, X, gamma=self.gamma_value_)
        K_mat = (K_mat + K_mat.T) / 2
        K_mat += 1e-8 * np.eye(N)
        
        alpha = cp.Variable(N)
        objective = cp.Maximize(r @ alpha - 0.5 * cp.quad_form(alpha, K_mat))
        constraints = [alpha >= 0, alpha <= self.C]
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.CLARABEL, verbose=False)
        self.alpha_ = np.array(alpha.value).flatten()
        return self
    
    def predict(self, X):
        K_mat = rbf_kernel(X, self.X_train_, gamma=self.gamma_value_)
        return K_mat @ self.alpha_

def predict_argmin(models, X):
    preds = np.column_stack([m.predict(X) for m in models])
    return np.argmin(preds, axis=1)

def avg_cost(y_true, y_pred, C):
    return np.mean(C[y_true, y_pred])

# Train full OSSVR
full_models = [OneSidedSVR(C=C_REG, gamma=GAMMA).fit(X_train, regret_train[:, k]) for k in range(K)]
y_pred_full = predict_argmin(full_models, X_test)
full_cost = avg_cost(y_test, y_pred_full, COST_MATRIX)
full_acc = accuracy_score(y_test, y_pred_full)

print(f"Full OSSVR — Cost: {full_cost:.4f}, Accuracy: {full_acc:.4f}")

Full OSSVR — Cost: 0.0741, Accuracy: 0.9444


---
## Ablation 1: Replace One-Sided Loss with Two-Sided (Standard) SVR Loss

**Component being ablated:** The **one-sided ε-insensitive loss** — the core novelty of the paper.

**Role in the full method:** The one-sided loss `max(0, r − f)` only penalises under-predictions of regret. Over-predictions are free because over-estimating the cost of a class is harmless — it simply makes the model more conservative about predicting that class. This asymmetry is the key design choice that the paper argues makes OSSVR superior to standard approaches. It appears in **Equation (5) of Section 3**.

**What the ablated version does:** We replace the one-sided loss with the standard **two-sided ε-insensitive loss** used in standard SVR: `max(0, |r − f| − ε)`, which penalises both over- and under-prediction equally. This removes the paper's core asymmetric treatment of errors.

In [9]:
# === Ablation 1: Two-Sided SVR (Standard epsilon-SVR) ===
from sklearn.svm import SVR

twosided_models = []
for k in range(K):
    svr = SVR(C=C_REG, kernel='rbf', gamma=GAMMA, epsilon=0.1)
    svr.fit(X_train, regret_train[:, k])
    twosided_models.append(svr)

y_pred_twosided = predict_argmin(twosided_models, X_test)
twosided_cost = avg_cost(y_test, y_pred_twosided, COST_MATRIX)
twosided_acc = accuracy_score(y_test, y_pred_twosided)

print(f"Ablation 1 (Two-Sided SVR) — Cost: {twosided_cost:.4f}, Accuracy: {twosided_acc:.4f}")
print(f"Full OSSVR (reference)     — Cost: {full_cost:.4f}, Accuracy: {full_acc:.4f}")
print(f"Cost difference: {twosided_cost - full_cost:+.4f}")

Ablation 1 (Two-Sided SVR) — Cost: 0.5370, Accuracy: 0.4630
Full OSSVR (reference)     — Cost: 0.0741, Accuracy: 0.9444
Cost difference: +0.4630


In [ ]:
# === Ablation 1: Visualisation ===
import os
# Robust path: works whether CWD is workspace root or partB/
_cwd = os.path.abspath('')
RESULTS_DIR = os.path.join(_cwd, 'results') if _cwd.endswith('partB') else os.path.join(_cwd, 'partB', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['Full OSSVR\n(One-Sided)', 'Ablation 1\n(Two-Sided SVR)']
costs = [full_cost, twosided_cost]
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(methods, costs, color=colors, edgecolor='black', width=0.5)
ax.set_ylabel('Average Misclassification Cost')
ax.set_title('Ablation 1: One-Sided vs. Two-Sided Loss')
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{cost:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ablation1_onesided_vs_twosided.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

Saved.


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_79846/3792384364.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Ablation 1 — Interpretation

Replacing the one-sided loss with the standard two-sided SVR loss tests whether the asymmetric treatment of errors is truly beneficial. The one-sided loss is the central algorithmic innovation of the paper: it allows the regressor to freely over-estimate the regret of any class (since over-estimating cost is a conservative, harmless behaviour) while strongly penalising under-estimation (which would cause the model to predict costly classes too often).

With the two-sided loss, the model spends capacity trying to *also* avoid over-predictions, which is unnecessary for cost-sensitive classification. This wastes model flexibility on an irrelevant objective. In practice, two-sided SVR may force the predicted regret values into a tighter band around the true values, but the argmin prediction rule only needs the *relative ordering* of regret predictions to be correct — not their absolute accuracy.

If the cost increases (worsens) under the two-sided loss, it confirms that the one-sided asymmetry is a meaningful contribution and not just a cosmetic difference. The magnitude of the change tells us how much of OSSVR's performance comes from this specific design choice versus other aspects (e.g., regression on cost values instead of classification).

---
## Ablation 2: Replace Regret Vectors with Raw Cost Vectors (No Shift)

**Component being ablated:** The **regret transformation** — subtracting the correct-class cost from each entry.

**Role in the full method:** The regret vector is defined as rₙ[k] = C[yₙ, k] − C[yₙ, yₙ]. Since C[y, y] = 0, this gives rₙ[k] = C[yₙ, k]. However, the conceptual significance is that the correct class is *anchored at zero*. If the cost matrix had non-zero diagonal entries (i.e., there was a baseline cost for correct classification), the regret transformation would subtract it out, normalising the targets. This anchoring ensures all regressors share a common reference point (0 = correct class), making the argmin comparison meaningful across classes. This appears in **Section 3, Equations (3)–(4)**.

**What the ablated version does:** Instead of regret vectors, we use **raw cost vectors without the shift**: we keep C[yₙ, k] as-is but also add a non-zero diagonal. Specifically, we add a constant offset to the entire cost matrix so the correct class has a non-zero cost, breaking the anchoring.

In [6]:
# === Ablation 2: Un-anchored cost vectors ===
# Add a class-dependent offset to break the zero-anchoring.
# This simulates using raw costs without the regret transformation.

OFFSET_MATRIX = COST_MATRIX.copy().astype(float)
# Add different offsets per row to break the zero anchor:
# Class 0 gets +3, Class 1 gets +7, Class 2 gets +1
offsets = np.array([3.0, 7.0, 1.0])
for i in range(K):
    OFFSET_MATRIX[i, :] += offsets[i]

print("Original regret matrix (correct class = 0):")
print(COST_MATRIX)
print(f"\nOffset cost matrix (correct class ≠ 0):")
print(OFFSET_MATRIX)
print(f"Diagonal: {np.diag(OFFSET_MATRIX)} (no longer zero)")

# Compute un-anchored targets
raw_train = OFFSET_MATRIX[y_train]

# Train OSSVR with un-anchored targets
raw_models = [OneSidedSVR(C=C_REG, gamma=GAMMA).fit(X_train, raw_train[:, k]) for k in range(K)]
y_pred_raw = predict_argmin(raw_models, X_test)
raw_cost = avg_cost(y_test, y_pred_raw, COST_MATRIX)
raw_acc = accuracy_score(y_test, y_pred_raw)

print(f"\nAblation 2 (Unanchored) — Cost: {raw_cost:.4f}, Accuracy: {raw_acc:.4f}")
print(f"Full OSSVR (reference) — Cost: {full_cost:.4f}, Accuracy: {full_acc:.4f}")
print(f"Cost difference: {raw_cost - full_cost:+.4f}")

Original regret matrix (correct class = 0):
[[ 0  1  5]
 [ 2  0  3]
 [10  1  0]]

Offset cost matrix (correct class ≠ 0):
[[ 3.  4.  8.]
 [ 9.  7. 10.]
 [11.  2.  1.]]
Diagonal: [3. 7. 1.] (no longer zero)

Ablation 2 (Unanchored) — Cost: 0.0556, Accuracy: 0.9630
Full OSSVR (reference) — Cost: 0.0741, Accuracy: 0.9444
Cost difference: -0.0185


In [7]:
# === Ablation 2: Visualisation ===
fig, ax = plt.subplots(figsize=(8, 5))
methods = ['Full OSSVR\n(Regret Vectors)', 'Ablation 2\n(Unanchored Costs)']
costs = [full_cost, raw_cost]
colors = ['#2ecc71', '#e67e22']
bars = ax.bar(methods, costs, color=colors, edgecolor='black', width=0.5)
ax.set_ylabel('Average Misclassification Cost')
ax.set_title('Ablation 2: Regret Vectors vs. Unanchored Cost Vectors')
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{cost:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ablation2_regret_vs_raw.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

Saved.


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_79846/4082791506.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Ablation 2 — Interpretation

Removing the regret transformation (i.e., not subtracting the correct-class cost to anchor it at zero) tests whether the target normalisation contributes to OSSVR's performance. The regret transformation is important for two reasons.

First, it creates a shared reference point across all classes: the correct class always has target 0, and all incorrect classes have targets > 0. Without this anchoring, different classes have different baseline cost levels, which means the K independent regressors are learning targets on incomparable scales. When we take argmin across their outputs at prediction time, we are comparing numbers that are not calibrated to the same zero point, potentially leading to systematic biases.

Second, anchoring at zero simplifies the regression task. With regret targets, the regressor only needs to learn the *additional* cost of each wrong class relative to the correct class. Without anchoring, it must learn absolute cost values, which adds unnecessary difficulty without providing any extra useful signal.

If cost increases in this ablation, it confirms that the regret transformation is a meaningful preprocessing step and not just a mathematical convenience. The effect may be smaller than Ablation 1 because the one-sided loss is a deeper algorithmic change, whereas the regret transformation is a target preprocessing step. However, even moderate degradation demonstrates that the target anchoring contributes to the reliability of the argmin prediction rule.

In [8]:
# === Summary Table ===
import pandas as pd

summary = pd.DataFrame({
    'Configuration': ['Full OSSVR', 'Ablation 1: Two-Sided SVR', 'Ablation 2: Unanchored Costs'],
    'Avg Cost': [full_cost, twosided_cost, raw_cost],
    'Accuracy': [full_acc, twosided_acc, raw_acc],
    'Cost Δ': [0, twosided_cost - full_cost, raw_cost - full_cost],
})

print("=" * 70)
print("ABLATION SUMMARY")
print("=" * 70)
print(summary.to_string(index=False))
print("=" * 70)

ABLATION SUMMARY
               Configuration  Avg Cost  Accuracy    Cost Δ
                  Full OSSVR  0.074074  0.944444  0.000000
   Ablation 1: Two-Sided SVR  0.537037  0.462963  0.462963
Ablation 2: Unanchored Costs  0.055556  0.962963 -0.018519
